# Reproducing Gonzalez-Cabrera et al. (2026) from dandiset 001955

**Paper.** Gonzalez-Cabrera C. et al. (2026). A VTA-pontine GABA pathway biases locomotor direction
via local and distal inhibition. bioRxiv preprint,
[doi:10.64898/2026.02.17.706301](https://doi.org/10.64898/2026.02.17.706301). The claims file
(`claims/gonzalez-cabrera-2026.yaml`) was written from the preprint's version 1 full text (main
text, Figure 1 to 4 legends, Methods, Supplementary Figure 1 to 4 legends), without the figure
panels, and the paper's code at github.com/CGonzC/VTAPnO_Locomotion_Analysis, tag
v1.0_manuscript_submission (commit 1ff5196).

**Dandiset.** [DANDI:001955](https://dandiarchive.org/dandiset/001955/0.261103.0758), version
0.261103.0758. The paper, a preprint of February 2026, cites no dandiset version (the dandiset was
created in August 2026). There are two published versions, 0.260828.0749 and 0.261103.0758. They
list the same 66 files, with the same asset identifiers, paths and sizes; the later one differs only
in metadata (a longer description that names the paper and the three experiment groups, the
contact's affiliation and ORCID, the funding statement). The most recent published version is
pinned, as the protocol says when the paper cites none; no file content differs.

**What the dandiset holds.** Raw broadband silicon-probe recordings of the acute (7 mice) and
chronic (4 subjects) experiments, with the Open Ephys ADC channels, and fiber photometry of VTA
dopamine neurons in 5 opsin and 3 control mice at 5, 10 and 20 Hz and continuous stimulation. The
ecephys files hold no spike-sorted units and no table of events; the juxtacellular, slice, tracing
and open-field experiments are not in the dandiset.

| Claims | Tested here |
|---|---|
| fig3b-photometry-continuous-only, fig3c-photometry-opsin-vs-control | tested |
| 9 acute and 6 chronic silicon-probe claims (fig3-optotag-counts to fig3k-da-early-timing, chronic-optotag-count to fig4f-paired-unit-profiles) | blocked, data missing: no sorted units |
| 11 juxtacellular and tracing claims (Figure 1, Supplementary Figure 1) | blocked, data missing |
| 5 slice claims (Figure 2a-c, Supplementary Figure 2b,c) | blocked, data missing |
| 8 open-field claims (Figure 2e-i, Supplementary Figure 2d-f) | blocked, data missing |

In [ ]:
import resource
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vtapno

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

started = time.time()
table = vtapno.assets()
print(f"dandiset {vtapno.DANDISET_ID} version {vtapno.DANDISET_VERSION}: {len(table)} files, "
      f"{table.size_gb.sum():.1f} GB")

## Data check

### What the files hold

The listing by group and subject, then, for every ecephys file, what it holds (metadata only).

In [ ]:
print(table.groupby("group").agg(files=("path", "size"), subjects=("subject", "nunique"), GB=("size_gb", "sum")).round(1).to_string())
print()
print(table.groupby(["group", "subject"]).size().rename("files").to_string())

In [ ]:
ephys_rows = table[table.group != "photometry"]
with ThreadPoolExecutor(8) as ex:
    contents = list(ex.map(vtapno.ecephys_contents, ephys_rows.url))
eph = pd.DataFrame(contents)
eph.insert(0, "file", ephys_rows.path.str.split("/").str[1].str.replace("_ecephys.nwb", "", regex=False).to_numpy())
eph.insert(1, "group", ephys_rows.group.to_numpy())
eph["hours"] = eph.samples / 30000 / 3600
show = ["file", "group", "subject_id", "recordings", "channels", "rates_hz", "hours", "adc_series", "units",
        "intervals", "processing", "analysis", "locations"]
print(eph[show].round(2).to_string(index=False))
print("\nFiles with a units table:", int(eph.units.sum()), "of", len(eph))
print("Files with any intervals table:", int((eph.intervals.str.len() > 0).sum()))
print("Files with any processing module:", int((eph.processing.str.len() > 0).sum()))
print("Files with any analysis content:", int((eph.analysis.str.len() > 0).sum()))
print("Electrode locations over all files:", sorted({l for ls in eph.locations for l in ls}))
print("Electrode table columns:", sorted({c for cols in eph.electrode_columns for c in cols}))
print("Session descriptions:", eph.session_description.unique().tolist())
print("ADC series descriptions (first part):", sorted({d for ds in eph.adc_descriptions for d in ds}))
tot = eph.groupby("group").agg(files=("file", "size"), recordings=("recordings", "sum"), hours=("hours", "sum"),
                               subjects=("subject_id", "nunique"))
tot["raw_GB_int16"] = [(eph[eph.group == g].samples * eph[eph.group == g].channels.str[0] * 2).sum() / 1e9 for g in tot.index]
print("\n", tot.round(1).to_string())

No ecephys file holds a units table, spike times, an intervals table or a processing module: each
holds the raw 30-kHz voltage of its Open Ephys recordings (64 channels acute, 32 chronic) and eight
ADC series per recording. The acute files are 7 subjects, as the paper's seven mice; the chronic
files are 4 subjects, where the paper (Methods, Chronic silicon-probe recordings; Results) has 3
mice. Nothing in the files says which recording is the baseline, the tagging, the 1-s light
protocol or a rotarod block, nor which direction the rod turned.

### The ADC channels

The ADC series carry no names beyond their number. Below, one acute recording and one chronic
recording are read whole (8 channels each) to see what they carry.

In [ ]:
acute_url = table[table.path.str.contains("Acute-VI-2024-04-17-12-12-40-experiment1")].url.iloc[0]
chronic_url = table[table.path.str.contains("chronic-DAT-2680_ses")].url.iloc[0]
for name, url, recs in (("acute-VI experiment1", acute_url, ["experiment1_recording4", "experiment1_recording7", "experiment1_recording1"]),
                        ("chronic-DAT-2680", chronic_url, ["experiment1_recording4", "experiment1_recording1"])):
    for rec in recs:
        p = vtapno.adc_pulses(url, rec)
        print(f"{name} {rec}:")
        print(p.round(3).to_string(index=False))
        print()

In each of these recordings one ADC channel carries 210 pulses of about 1 ms at 0.2-s intervals,
which matches the tagging protocol (Methods, Acute silicon-probe recordings: "1 ms light pulses at
5Hz ... 210 pulses in total"); it is ADC1 in some recordings and ADC2 in others, and which channel
is the 470-nm and which the 635-nm laser is not recorded. Other channels carry analog signals or
irregular crossings whose meaning the files do not give. The light timing is therefore in the
files, but unlabelled.

### How long re-sorting would take

The paper sorted the raw data with Kilosort (version not given) and curated the clusters by hand in
Phy 2 (Methods, Electrophysiology acquisition and spike sorting). Streaming 20 s of one 64-channel
recording measures the read rate:

In [ ]:
mb, sec = vtapno.time_raw_read(acute_url, "Ephys_experiment1_recording4", 20.0)
raw_gb = float(tot.raw_GB_int16.sum())
print(f"read {mb:.0f} MB in {sec:.1f} s, {mb / sec:.0f} MB/s")
print(f"all raw ecephys: {raw_gb:.0f} GB uncompressed, {tot.hours.sum():.1f} h of recording; "
      f"streaming alone at this rate: {raw_gb * 1000 / (mb / sec) / 3600:.1f} h")

### Photometry

Each photometry file is one mouse and one condition. The two signals are loaded, reduced to dF/F as
the paper's code does, and measured under every reading (helper `vtapno.process_photometry`).

In [ ]:
t0 = time.time()
recs = vtapno.load_all_photometry(table)
print(f"processed {len(recs)} photometry files in {time.time() - t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
meta = pd.DataFrame([r["meta"] for r in recs])
meta["trials"] = [len(r["stim"]) for r in recs]
meta["interval_s"] = [sorted(set((r["stim"].stop - r["stim"].start).round(3))) for r in recs]
meta["onset_gap_s"] = [float(np.median(np.diff(r["stim"].start))) for r in recs]
meta["first_onset_s"] = [float(r["stim"].start.iloc[0]) for r in recs]
meta["last_stop_s"] = [float(r["stim"].stop.iloc[-1]) for r in recs]
meta["events"] = [sorted(set(r["stim"].event)) for r in recs]
meta["animal_in_notes"] = meta.notes.str.extract(r"Animal: (\d+)")[0]
cols = ["subject", "group", "condition", "session_id", "animal_in_notes", "age", "sex", "session_start", "rate_hz",
        "duration_s", "nan_frac_ca", "trials", "interval_s", "onset_gap_s", "first_onset_s", "last_stop_s", "events",
        "unit_ca", "same_timestamps", "r2", "slope", "floored_frac", "md5_ca"]
print(meta[cols].sort_values(["group", "subject", "condition"]).round(4).to_string(index=False))

In [ ]:
print("Mice by group:", meta.groupby("group").subject.nunique().to_dict(), "(paper: opsin 5, control 3)")
print("Files per mouse and condition:")
print(pd.crosstab(meta.subject, meta.condition).to_string())
print("\nSampling rate (Hz), min and max over files:", round(meta.rate_hz.min(), 2), round(meta.rate_hz.max(), 2))
print("Largest deviation of a sample interval from the file's median (ratio):",
      round(max(abs(a - 1) for r in meta.dt_ratio_range for a in r), 9))
print("GCaMP and isosbestic share one timestamp array in every file:", bool(meta.same_timestamps.all()))
print("Missing samples per file, fraction: min", meta.nan_frac_ca.min().round(5), "max", meta.nan_frac_ca.max().round(5),
      "; same positions in both signals:", bool((meta.nan_frac_ca == meta.nan_frac_iso).all()))
print("Signal ranges (a.u.): GCaMP", round(min(r[0] for r in meta.ca_range), 4), "to", round(max(r[1] for r in meta.ca_range), 4),
      "; isosbestic", round(min(r[0] for r in meta.iso_range), 4), "to", round(max(r[1] for r in meta.iso_range), 4))
print("Identical raw GCaMP arrays between files:", int(meta.md5_ca.duplicated().sum()))
print("Interval lengths by condition (s):",
      {c: sorted({x for xs in meta[meta.condition == c].interval_s for x in xs}) for c in vtapno.FREQUENCIES})
print("Files whose isosbestic fit has R^2 under 0.30 (the code's warning level):", int((meta.r2 < 0.30).sum()), "of", len(meta),
      "; R^2 range", meta.r2.min().round(3), "to", meta.r2.max().round(3))
print("Samples where the dF/F denominator was floored: max fraction", meta.floored_frac.max())
odd = meta[meta.interval_s.str.len() > 1]
for _, r in odd.iterrows():
    st = next(x["stim"] for x in recs if x["meta"]["path"] == r.path)
    print(f"\n{r.subject} {r.condition}: intervals of more than one length")
    print(st.assign(length=(st.stop - st.start).round(3)).head(4).round(3).to_string(index=False))
# Session start times against durations: does a file start before the previous file of the same mouse ends?
meta["start_ts"] = pd.to_datetime(meta.session_start, utc=True, format="ISO8601")
over = []
for s, g in meta.sort_values("start_ts").groupby("subject"):
    g = g.sort_values("start_ts")
    gaps = (g.start_ts.diff().dt.total_seconds()).to_numpy()[1:]
    durs = g.duration_s.to_numpy()[:-1]
    over.append({"subject": s, "files": len(g), "starts_before_previous_ends": int((gaps < durs).sum()),
                 "min_gap_s": float(np.min(gaps)) if len(gaps) else np.nan})
print("\nSession start times against file durations:")
print(pd.DataFrame(over).round(1).to_string(index=False))

The files match the description: 5 opsin (`photometry-exp-1` to `-5`) and 3 control mice, each with
one file for each of the four conditions, so 32 files; the paper has 5 opsin and 3 control mice.
GCaMP and isosbestic share one timestamp array at about 12 kHz with regular intervals, and both are
in arbitrary units. Each file has 23 stimulation intervals 13 s apart, except `photometry-exp-5`
with 13. The interval lengths fit the conditions: 3.0 s for continuous light and 2.80, 2.90 and
2.95 s for the 5, 10 and 20 Hz trains (the train's last pulse ending before 3 s). One file,
`photometry-exp-5` at 5 Hz, also has two 3.0-s intervals 8 s apart at the start of the recording;
the paper's code would use every event in the file, so all intervals are used, and the result
without the two is shown as a variant. No two files hold the same raw signal. The isosbestic fit
explains little of the GCaMP variance in 12 of the 32 files (R² under the code's 0.30 warning
level; the R² is computed over the whole recording, light responses included), which the code
warns about but does not act on. The dF/F denominator floor is never reached.

The `session_start_time` of a file is often earlier than the end of the previous file of the same
mouse (counted above), so these times cannot all be recording starts; they look like export times.
No analysis uses them. Ages are given as a range (P16W/P20W) and sex as M for every mouse.

### Clock of the stimulation table

The stimulation intervals and the signals must share a clock. The mean change in dF/F (0-3 s minus
-1.0 to -0.1 s) is computed at the table's onsets and at onsets shifted by half the 13-s cycle
(±6.5 s), where no light is on.

In [ ]:
clk = pd.DataFrame([{"subject": r["meta"]["subject"], "group": r["meta"]["group"], "condition": r["meta"]["condition"],
                     "true": r["contrast"][0.0], "shift_+6.5": r["contrast"][6.5], "shift_-6.5": r["contrast"][-6.5]}
                    for r in recs])
print("Mean dF/F change, Stim minus Pre, at the true and the shifted onsets (dF/F units):")
print(clk.pivot_table(index=["group", "subject"], columns="condition", values=["true", "shift_+6.5"]).round(4).to_string())
for (g, c), d in clk.groupby(["group", "condition"]):
    print(f"{g:8s} {c:5s} true {d['true'].mean():+.4f}  shifted +6.5 {d['shift_+6.5'].mean():+.4f}  -6.5 {d['shift_-6.5'].mean():+.4f}")

In [ ]:
cont = clk[(clk.group == "opsin") & (clk.condition == "Cont")].set_index("subject")
cont["true_above_both_shifts"] = cont["true"] > cont[["shift_+6.5", "shift_-6.5"]].max(axis=1)
print(cont[["true", "shift_+6.5", "shift_-6.5", "true_above_both_shifts"]].round(4).to_string())

In each opsin mouse under continuous light the change at the true onsets is positive and larger
than at either shifted set of onsets (printed above; `photometry-exp-5` is the weakest), and the
mean over opsin mice is 0.0410 at the true onsets against -0.0010 and -0.0015 shifted, so the
stimulation table and the signal share a clock in those files. In the control mice and at the train frequencies there is little response to
contrast with, so for those files the shared clock is consistent with the data (the same table
format and the same acquisition), not shown.

## fig3b-photometry-continuous-only

> In opsin mice, VTA-PnO stimulation raises the DA population calcium signal above its
> pre-stimulus level with continuous light, but not with 5, 10 or 20 Hz trains.

Criterion (claims file): per mouse and condition, the trial-averaged z-scored trace; Pre = -1.0 to
-0.1 s, Stim = 0-3 s; a two-way ANOVA epoch by frequency (repeated measures on both factors, or
not); post hoc paired Pre against Stim per frequency, Benjamini-Hochberg over the four. Reproduced
when under one reading continuous gives p < 0.0001 with Stim higher and 5, 10 and 20 Hz each give
p > 0.05. The readings the criterion names are all run: z baseline (Methods -1.0 to -0.1 s, code
-3.0 to -0.1 s), z form (mean/SD, median/MAD), epoch value (mean, area) and ANOVA (repeated, not).
Two details the claims file does not fix are also run: the sampling rate after the "downsampled"
step of the Methods (100 Hz, a standard choice, or the native 12 kHz, as the code), and whether the
two stray intervals of `photometry-exp-5` at 5 Hz are used (all intervals, as the code).

The post hoc test is not fixed by the claims file beyond "paired Pre against Stim". The legend
says "two-way ANOVA with FDR-corrected post hoc comparisons" and the Methods (Quantification and
statistical analysis) name GraphPad Prism v10, whose post hoc comparisons after a two-way ANOVA
use the ANOVA's pooled error by default. Both forms are run: a separate paired t-test per
frequency, and the contrast on the pooled error (helper `within_opsin_tests`). This doubles the
readings to 128.

In [ ]:
rows = []
for r in recs:
    for onset_set, ms in r["measures"].items():
        for m in ms:
            rows.append({"subject": r["meta"]["subject"], "group": r["meta"]["group"], "condition": r["meta"]["condition"],
                         "onsets": onset_set, **{k: v for k, v in m.items() if k != "trace"}})
measures = pd.DataFrame(rows)
reading_keys = ["onsets", "rate", "baseline", "form", "value"]
print("Readings:", measures.groupby(reading_keys).ngroups, "; trials dropped by the code's baseline guard, total over files:",
      int(measures[(measures.onsets == "all") & (measures.rate == "100Hz") & (measures.value == "mean")].groupby("baseline").dropped.sum().get("code", 0)))
main = measures[(measures.onsets == "all") & (measures.rate == "100Hz") & (measures.baseline == "methods")
                & (measures.form == "mean_sd") & (measures.value == "mean")]
print("\nMain reading (Methods baseline, mean/SD z, 100 Hz, epoch means, all intervals), per mouse:")
print(main.pivot_table(index=["group", "subject"], columns="condition", values=["pre", "stim"])
      .reindex(columns=pd.MultiIndex.from_product([["pre", "stim"], vtapno.FREQUENCIES])).round(3).to_string())

In [ ]:
def run_3b(measures):
    out = []
    for key, d in measures.groupby(reading_keys):
        for anova, posthoc in [(a, p) for a in ("rm", "plain") for p in ("separate", "pooled")]:
            res = vtapno.within_opsin_tests(d, anova, posthoc)
            row = dict(zip(reading_keys, key), anova=anova, posthoc=posthoc, interaction_p=res["interaction_p"])
            for c in vtapno.FREQUENCIES:
                row[f"diff_{c}"] = res["mean_diff"][c]
                row[f"p_{c}"] = res["p_bh"][c]
            row["holds"] = bool(row["p_Cont"] < 1e-4 and row["diff_Cont"] > 0
                                and all(row[f"p_{c}"] > 0.05 for c in ("5hz", "10hz", "20hz")))
            row["cont_p05"] = bool(row["p_Cont"] < 0.05 and row["diff_Cont"] > 0)
            row["trains_ns"] = int(sum(row[f"p_{c}"] > 0.05 for c in ("5hz", "10hz", "20hz")))
            out.append(row)
    return pd.DataFrame(out)

r3b = run_3b(measures)
pcols = [f"p_{c}" for c in vtapno.FREQUENCIES]
dcols = [f"diff_{c}" for c in vtapno.FREQUENCIES]
fmt = r3b.copy()
for c in pcols + ["interaction_p"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:.2g}")
for c in dcols:
    fmt[c] = fmt[c].map(lambda x: f"{x:.3f}")
print("Post hoc Pre against Stim (paired t, BH over 4 frequencies) per reading; diff = mean Stim - Pre over opsin mice:")
print(fmt.to_string(index=False))
print(f"\nReadings under which the criterion holds: {int(r3b.holds.sum())} of {len(r3b)}")
summ = r3b.groupby(["posthoc", "anova"]).agg(readings=("holds", "size"), holds=("holds", "sum"),
                                             p_Cont_min=("p_Cont", "min"), p_Cont_max=("p_Cont", "max"),
                                             trains_all_ns=("trains_ns", lambda x: int((x == 3).sum())),
                                             p_20hz_min=("p_20hz", "min"))
print("\nBy post hoc form and ANOVA:")
print(summ.to_string(float_format=lambda x: f"{x:.2g}"))
print(f"Continuous significant at p < 0.05 with Stim higher: {int(r3b.cont_p05.sum())} of {len(r3b)}; "
      f"at p < 0.0001: {int((r3b.p_Cont < 1e-4).sum())}")
print("Smallest continuous p over readings:", f"{r3b.p_Cont.min():.2g}", "; largest:", f"{r3b.p_Cont.max():.2g}")
print("Readings with all three train frequencies p > 0.05:", int((r3b.trains_ns == 3).sum()), "of", len(r3b))
for c in ("5hz", "10hz", "20hz"):
    print(f"  {c}: smallest p {r3b[f'p_{c}'].min():.3g}, readings with p < 0.05: {int((r3b[f'p_{c}'] < 0.05).sum())}")

In [ ]:
from scipy import stats
print("Main reading, continuous light: Stim - Pre per opsin mouse")
d = main[(main.group == "opsin") & (main.condition == "Cont")]
print((d.set_index("subject").stim - d.set_index("subject").pre).round(3).to_string())
t = stats.ttest_rel(d.stim, d.pre)
print(f"paired t({len(d) - 1}) = {t.statistic:.2f}, uncorrected p = {t.pvalue:.2g}")
print("\nStim - Pre per opsin mouse at each train frequency (main reading):")
for c in ("5hz", "10hz", "20hz"):
    d = main[(main.group == "opsin") & (main.condition == c)]
    print(c, (d.stim - d.pre).round(3).tolist())
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
lo, hi = np.inf, -np.inf
for r in recs:
    m = next(x for x in r["measures"]["all"] if x["rate"] == "100Hz" and x["baseline"] == "methods" and x["form"] == "mean_sd" and x["value"] == "mean")
    grid, tr = m["trace"]
    ax = axes[vtapno.FREQUENCIES.index(r["meta"]["condition"])]
    ax.plot(grid, tr, color="C3" if r["meta"]["group"] == "opsin" else "0.5", lw=0.8)
    lo, hi = min(lo, np.nanmin(tr)), max(hi, np.nanmax(tr))
for ax, c in zip(axes, vtapno.FREQUENCIES):
    ax.axvspan(0, 3, color="C0", alpha=0.1); ax.set_title(c); ax.set_xlabel("s from light onset")
axes[0].set_ylabel("z (baseline -1.0 to -0.1 s)")
plt.tight_layout(); plt.show()
print(f"Plotted traces (opsin red, control grey) span {lo:.2f} to {hi:.2f} z; all points within the axes.")

**Result: reproduced, resting on the assumption that the post hoc comparisons use the ANOVA's
pooled error.** With that error, continuous light raises the signal with BH-corrected p from
3.8e-06 to 2.9e-04 over the 64 pooled readings, and 5, 10 and 20 Hz are not significant in any
of them (20 Hz p 0.44 or more). The criterion's p < 0.0001 for continuous light holds in 52 of the
64: all 32 with the ordinary two-way ANOVA and 20 of 32 with the repeated-measures error, which
fail only with the code's -3.0 s z baseline. With a separate paired t-test per frequency, the
continuous effect is not significant after correction in most readings (BH p 0.048 to 0.11),
though Stim exceeds Pre in each of the 5 opsin mice (1.9 to 15.0 z under the main reading,
uncorrected p 0.04), and in 12 of those 64 readings a train frequency (10 or 20 Hz) also
reaches p < 0.05; under that form
the claim would be at most partially reproduced.

## fig3c-photometry-opsin-vs-control

> The stimulation-evoked change in DA calcium signal (Stim minus Pre) is larger in opsin mice than
> in controls for continuous light only, not for 5-20 Hz.

Criterion (claims file): Δ = Stim - Pre per mouse and frequency, under the readings of
fig3b-photometry-continuous-only; a two-way ANOVA group by frequency (frequency repeated within
mouse, or not); post hoc unpaired opsin against control per frequency, Benjamini-Hochberg over the
four. Reproduced when under one reading continuous gives p < 0.05 with opsin larger and 5, 10 and
20 Hz each give p > 0.05. The post hoc test is run in the two forms of
fig3b-photometry-continuous-only: a separate unpaired t-test per frequency, and the contrast on
the pooled within-cell error (helper `opsin_vs_control_tests`).

In [ ]:
def run_3c(measures):
    out = []
    for key, d in measures.groupby(reading_keys):
        for anova, posthoc in [(a, p) for a in ("rm", "plain") for p in ("separate", "pooled")]:
            res = vtapno.opsin_vs_control_tests(d, anova, posthoc)
            row = dict(zip(reading_keys, key), anova=anova, posthoc=posthoc, interaction_p=res["interaction_p"])
            for c in vtapno.FREQUENCIES:
                row[f"diff_{c}"] = res["mean_diff"][c]
                row[f"p_{c}"] = res["p_bh"][c]
            row["holds"] = bool(row["p_Cont"] < 0.05 and row["diff_Cont"] > 0
                                and all(row[f"p_{c}"] > 0.05 for c in ("5hz", "10hz", "20hz")))
            row["trains_ns"] = int(sum(row[f"p_{c}"] > 0.05 for c in ("5hz", "10hz", "20hz")))
            out.append(row)
    return pd.DataFrame(out)

r3c = run_3c(measures)
fmt = r3c.copy()
for c in pcols + ["interaction_p"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:.2g}")
for c in dcols:
    fmt[c] = fmt[c].map(lambda x: f"{x:.3f}")
print("Post hoc opsin against control (Student t, BH over 4) per reading; diff = mean delta opsin - control:")
print(fmt.to_string(index=False))
print(f"\nReadings under which the criterion holds: {int(r3c.holds.sum())} of {len(r3c)}")
summ = r3c.groupby(["posthoc", "anova"]).agg(readings=("holds", "size"), holds=("holds", "sum"),
                                             p_Cont_min=("p_Cont", "min"), p_Cont_max=("p_Cont", "max"),
                                             trains_all_ns=("trains_ns", lambda x: int((x == 3).sum())),
                                             p_20hz_min=("p_20hz", "min"))
print("\nBy post hoc form and ANOVA:")
print(summ.to_string(float_format=lambda x: f"{x:.2g}"))
print(f"Continuous p < 0.05 with opsin larger: {int(((r3c.p_Cont < 0.05) & (r3c.diff_Cont > 0)).sum())} of {len(r3c)}; "
      f"p range {r3c.p_Cont.min():.2g} to {r3c.p_Cont.max():.2g}")
for c in ("5hz", "10hz", "20hz"):
    print(f"  {c}: smallest p {r3c[f'p_{c}'].min():.3g}, readings with p < 0.05: {int((r3c[f'p_{c}'] < 0.05).sum())}")

In [ ]:
d = main.assign(delta=main.stim - main.pre)
print("Main reading: delta (Stim - Pre, z) per mouse")
print(d.pivot_table(index=["group", "subject"], columns="condition", values="delta")[vtapno.FREQUENCIES].round(3).to_string())
print("\nGroup means:")
print(d.groupby(["group", "condition"]).delta.agg(["mean", "std", "count"]).unstack(0).round(3).to_string())

**Result: reproduced, resting on the same assumption about the post hoc error.** With the pooled
error, the opsin change is larger than the control change for continuous light in all 64 pooled
readings (BH p 0.00074 to 0.0028; the paper reports 0.002), and 5, 10 and 20 Hz are not
significant in any (20 Hz p 0.66 or more). With a separate unpaired t-test per frequency, three
controls against five opsin mice, the continuous difference is not significant in any reading
(BH p 0.098 to 0.19), so the claim is not reproduced under that form. Under the main reading the
continuous change is 7.10 z in opsin mice against 0.12 in controls.

## Blocked claims

The other 39 claims are not tested. Their data are not in the dandiset:

- The acute (9 claims) and chronic (6 claims) silicon-probe claims rest on spike-sorted, curated
  and opto-tagged units. The ecephys files hold only raw voltage and unlabelled ADC channels (Data
  check). The paper's units come from Kilosort, of unstated version, followed by manual curation in
  Phy 2, which no rerun can repeat, and the light colour of each ADC channel and the rotarod
  direction of each block are not recorded. The streaming estimate above is for reading the data
  alone, before any sorting.
- The juxtacellular, tracing, slice and open-field claims (24) use experiments the dandiset does
  not hold. The Zenodo record that the dandiset cites (doi:10.5281/zenodo.21390830) holds
  figure-level source tables, histology images of Figure 1a, and eight open-field videos (MP4)
  with no DeepLabCut tracking or model; the paper's own Zenodo record (doi:10.5281/zenodo.18661700)
  holds the figure PDFs and the key resources table. Neither holds the recordings or the tracking.

## Summary

| Claim | Outcome |
|---|---|
| fig3b-photometry-continuous-only | reproduced, resting on an assumption (post hoc on the ANOVA's pooled error) |
| fig3c-photometry-opsin-vs-control | reproduced, resting on the same assumption |
| 15 acute and chronic silicon-probe claims | blocked, data missing (no sorted units) |
| 24 juxtacellular, tracing, slice and open-field claims | blocked, data missing |

How the analysis differs from the paper's. The photometry preprocessing is the paper's code; the
downsampling rate is a standard choice (100 Hz; native rate as a variant, with the same outcomes).
The post hoc test after the two-way ANOVA is not stated; the outcome rests on reading it as
GraphPad Prism's default, a contrast on the ANOVA's pooled error. With separate t-tests per
frequency, neither photometry claim holds, because the continuous effect, positive in all five
opsin mice, varies in size from 1.9 to 15.0 z, and with five mice its separate test is weak.

What was found about the dataset:

- The ecephys files (25 acute, 9 chronic; 161 GB) hold only raw voltage and unnamed ADC channels:
  no spike-sorted units, no tagging results, no event tables, no labels for the light colour of
  the laser TTLs or for the rotarod direction and blocks. None of the paper's 15 silicon-probe
  findings can be tested from them without redoing the spike sorting and its manual curation.
- The chronic files are from 4 subjects; the paper has 3 mice.
- Electrode locations are "unknown" in every ecephys file.
- In the photometry files, `session_start_time` is often earlier than the end of the previous file
  of the same mouse, so it is not the recording start.
- One photometry file (`photometry-exp-5`, 5 Hz) has two extra 3.0-s intervals before the regular
  train; the outcomes are the same with and without them.
- The isosbestic signal explains little of the GCaMP signal in 12 of 32 files (R² under 0.30,
  the warning level of the paper's own code).
- The juxtacellular, tracing, slice and open-field data are not in the dandiset; the Zenodo
  record it cites holds figure-level tables and open-field videos without tracking.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")